<a href="https://colab.research.google.com/github/gitmystuff/DTSC4050/blob/main/11_Regression_II/Your_Name_Regression_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regression II

Your Name

## Getting Started

* Colab - get notebook from gitmystuff DTSC4050 repository
* Save a Copy in Drive
* Remove Copy of
* Edit name
* Clean up Colab Notebooks folder
* Submit shared link


## Why Regression

Both **Linear Regression** and **Analysis of Variance (ANOVA)** are tools designed to help us understand relationships between variables. In fact, mathematically speaking, they are first cousins—both are variations of the **General Linear Model**.

However, they are built for different types of data and answer slightly different questions.

---

## Why We Use Regression (and Linear Regression)

We use regression analysis when we want to understand how a change in one or more **independent variables (predictors)** affects a **dependent variable (outcome)**.

Specifically, **Linear Regression** is used when we want to model the relationship between a continuous outcome and one or more predictors by fitting a straight line to the data.

We use it for three main reasons:

1. **Description & Effect Size:** To see *if* a relationship exists and *how strong* it is. (e.g., "For every additional year of experience, does a data scientist's salary increase, and by how much?")
2. **Prediction:** To predict a continuous outcome for new, unseen data points based on the historical trend line.
3. **Control:** To isolate the impact of a specific variable while statistically holding other confounding variables constant.

Mathematically, a simple linear regression model is expressed as:

$$y = \beta_0 + \beta_1 x + \epsilon$$

Where $y$ is the continuous dependent variable, $x$ is the predictor, $\beta_0$ is the intercept, $\beta_1$ is the slope (coefficient), and $\epsilon$ is the error term.

---

## Linear Regression vs. ANOVA: The Key Contrasts

While both techniques look for relationships, the fundamental distinction lies in the **nature of the predictor variables** and **what you intend to do with the results**.

| Feature | Linear Regression | ANOVA (Analysis of Variance) |
| --- | --- | --- |
| **Predictor Variables ($X$)** | Typically **continuous** (numerical scales like age, temperature, or test scores), though it can handle dummy-coded categorical variables. | Strictly **categorical** (groups, treatments, or factors—e.g., Control vs. Drug A vs. Drug B). |
| **Outcome Variable ($Y$)** | **Continuous** | **Continuous** |
| **Core Objective** | To **predict** an outcome and estimate the **rate of change** (the slope $\beta_1$) per unit increase in $X$. | To **compare group means** and determine if at least one group performs significantly differently from the others. |
| **Primary Metric** | Focuses on coefficients ($\beta$ weights), $R^2$ (variance explained), and $t$-tests for slope significance. | Focuses on the $F$-statistic, partitioning variance into **Between-Group** vs. **Within-Group** variance. |
| **Output Visualization** | A scatter plot with a fitted **regression line**. | Bar charts or box plots showing **differences in group means**. |

### A Conceptual Scenario

To make the contrast concrete, imagine you are studying student performance:

* **You would use Linear Regression** if you want to see how the *exact number of hours spent studying* (continuous $X$) predicts a student's final exam score (continuous $Y$). The output gives you a slope: for every 1 hour of study, the exam score increases by, say, 2.5 points.
* **You would use ANOVA** if you divided students into three distinct treatment groups: *No Study Guide, Standard Study Guide, and Interactive App* (categorical $X$). You want to know if the average exam scores (continuous $Y$) differ significantly across these three distinct categories.

### Summary of the Mathematical Overlap

Under the hood, if you run a One-Way ANOVA with two groups, it yields the exact same $p$-value as a simple linear regression where the categorical predictor is coded as a 0 or 1 dummy variable. The main difference is perspective: ANOVA asks *"Are these group means different?"* while regression asks *"What is the targeted effect size of moving from one group to the next?"*

## F-Statistic

In multiple linear regression, the **$F$-value** (or $F$-statistic) evaluates the overall statistical significance of the entire regression model.

While individual $t$-tests look at each predictor in isolation, the $F$-test steps back to ask a holistic question: **"Does this collective group of independent variables tell us anything statistically meaningful about the dependent variable, or are we just fitting noise?"**

---

### The Null and Alternative Hypotheses

The $F$-test evaluates the model against a strict null hypothesis:

* **Null Hypothesis ($H_0$):** $\beta_1 = \beta_2 = \dots = \beta_p = 0$
*(All slope coefficients are exactly zero. The model has zero predictive power, and the best guess for any data point is simply the sample mean $\bar{y}$.)*
* **Alternative Hypothesis ($H_a$):** At least one $\beta_i \neq 0$
*(At least one predictor in the model has a linear relationship with the outcome variable.)*

---

### Interpreting the $F$-Value

* **An $F$-value close to 1:** Indicates that the model explains about as much variance as you would expect by pure random chance. You will fail to reject the null hypothesis.
* **A large $F$-value:** Indicates that the variance explained by the model vastly outweighs the random error variance.

To determine if the $F$-value is large enough to be meaningful, it is mapped to an **$F$-distribution** curve based on its two distinct degrees of freedom: $(p, n - p - 1)$. This mapping yields the **overall $p$-value** of the model. If this $p$-value falls below a chosen significance threshold (e.g., $\alpha = 0.05$), you reject the null hypothesis and conclude that your model is statistically viable.

---

### Why Do We Need the $F$-Value if We Have $t$-Tests?

The $F$-value protects you from **Type I errors (false positives) driven by multiple comparisons**.

If you build a large model with 20 completely random, meaningless predictors, each individual $t$-test has a $5\%$ chance of accidentally looking significant at the $\alpha = 0.05$ level. You would expect roughly one variable to show up as a false positive by pure luck.

The $F$-test accounts for the entire set of predictors simultaneously. If all 20 variables are noise, the $F$-test evaluates the collective model, yields a non-significant result, and flashes a warning light before you mistakenly interpret an accidental, isolated $t$-test success.

## The Fertilizer Example

Recall the example of an agricultural researcher testing the effectiveness of three different fertilizers (A, B, and C) on the height of a specific type of plant. You randomly assign 15 plants to three groups:

* **Group 1 (Fertilizer A):** [10, 12, 14, 11, 13] - These are the heights (in centimeters) of the plants treated with Fertilizer A.
* **Group 2 (Fertilizer B):** [18, 20, 19, 21, 22] - Heights of plants treated with Fertilizer B.
* **Group 3 (Fertilizer C):** [5, 7, 6, 8, 9] - Heights of plants treated with Fertilizer C.

**Question:** Do three fertilizers (A, B, C) produce different average plant heights?


## F-Distribution

### Variance Between Groups (VBG)

* $SSG = \sum_{i=1}^{k} n_i (\bar{x}_i - \bar{x})^2$
* $VBG = SSG/ngroups - 1$

In [ ]:
import numpy as np

def calculate_vbg(groups):
    """
    Calculates the Mean Square Between (MSB) for ANOVA.

    Args:
        groups: A list of lists, where each inner list represents a group.

    Returns:
        The MSB value.
    """

    k = len(groups)  # Number of groups
    group_means = [np.mean(group) for group in groups]
    all_data = np.concatenate(groups)
    overall_mean = np.mean(all_data)
    n_values = [len(group) for group in groups]

    ssg = sum(n * (group_mean - overall_mean) ** 2 for n, group_mean in zip(n_values, group_means))
    df_b = k - 1
    print(f"SSB: {ssg}")
    print(f"Degrees of Freedom Between: {df_b}")

    vbg = ssg / df_b if df_b > 0 else 0  # Avoid division by zero

    return vbg

# Example usage:
group1 = [10, 12, 14, 11, 13]
group2 = [18, 20, 19, 21, 22]
group3 = [5, 7, 6, 8, 9]

groups = [group1, group2, group3]

vbg = calculate_vbg(groups)
print(f"Variance Between Groups: {vbg}")

### Variance Within Groups

* SSE = $\sum_{i=1}^{k} \sum_{j=1}^{n_i} (x_{ij} - \bar{x}_i)^2$
* VWG = SSE / nrows-ngroups

In [ ]:
import numpy as np

def calculate_variance_within_groups(groups):
    """
    Calculates the variance within groups (Mean Square Within - MSW) for ANOVA.

    Args:
        groups: A list of lists, where each inner list represents a group.

    Returns:
        The MSW value.
    """

    sse = 0  # Sum of Squares Within
    df_within = 0  # Degrees of freedom within groups

    for group in groups:
        group_array = np.array(group)
        group_mean = np.mean(group_array)
        sse += np.sum((group_array - group_mean) ** 2)
        df_within += len(group) - 1

    vwg = sse / df_within if df_within > 0 else 0 #Avoid division by zero
    print(f"SSE: {sse}")
    print(f"Degrees of Freedom Within: {df_within}")

    return vwg

vwg = calculate_variance_within_groups(groups)
print(f"Variance Within Groups (MSW): {vwg}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# Degrees of freedom
df_between = 2
df_within = 12

# F-statistic (example value)
f_statistic = 86

# Significance level (alpha)
alpha = 0.05

# Calculate critical value
critical_value = stats.f.ppf(1 - alpha, df_between, df_within)

# Generate F-distribution curve
x = np.linspace(0, 10, 1000)
f_distribution = stats.f.pdf(x, df_between, df_within)

# Plot the F-distribution
plt.figure(figsize=(8, 6))
plt.plot(x, f_distribution, label=f"df1 = {df_between}, df2 = {df_within}")

# Plot the F-statistic
plt.axvline(f_statistic, color='blue', linestyle='-', label="f statistic")

# Plot the critical value
plt.axvline(critical_value, color='red', linestyle='-', label="critical value")

# Add labels and title
plt.title(f"Critical value: {critical_value:.3f}")
plt.legend()
plt.show()

## GLMs

A **Generalized Linear Model (GLM)** is a flexible mathematical framework that extends ordinary linear regression to data where the residual errors are *not* normally distributed, and the relationship between the predictors and the outcome is non-linear.

Ordinary linear regression assumes your target variable ($Y$) is continuous and its errors follow a neat, bell-shaped normal curve. But in the real world, you often encounter binary outcomes (e.g., pass/fail), counts (e.g., number of website visits), or highly skewed data. GLMs solve this by unifying these different data types under one mathematical roof.

---

## The Three Components of a GLM

Every GLM consists of three structural pieces:

### 1. The Random Component

This specifies the conditional probability distribution of the dependent variable ($Y$). Instead of being restricted to a normal distribution, GLMs can accommodate any distribution from the **Exponential Family**, which includes:

* **Normal / Gaussian** (for standard continuous data)
* **Bernoulli / Binomial** (for binary or proportion data)
* **Poisson** (for count data)
* **Gamma** (for highly skewed positive continuous data)

### 2. The Systematic Component

This is the linear combination of your independent variables ($X$), often referred to as the **linear predictor** ($\eta$). It looks exactly like the right side of a standard regression equation:

$$\eta = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_k x_k$$

### 3. The Link Function $g(\cdot)$

This is the crucial bridge of the GLM. The link function maps the expected value of the target variable, $E(Y)$, to the linear predictor ($\eta$).

$$g(E(Y)) = \eta$$

Instead of forcing the relationship between $X$ and $Y$ to be a straight line, the link function linearizes the connection. For instance:

* In **Logistic Regression**, the logit link function maps probabilities bounded between 0 and 1 to an infinite linear scale.
* In **Poisson Regression**, the log link function maps positive event counts to a linear scale.

---

## Why Use GLMs?

* **Flexibility:** They allow you to apply familiar linear regression principles (like coefficients, hypothesis testing, and ANOVA-like deviance tables) to non-linear data structures.
* **Bounded Predictions:** If you model a binary outcome with standard linear regression, your model might predict a nonsensical probability of 120% or -15%. A GLM uses the link function to keep predictions strictly within valid boundaries.
* **Handles Heteroscedasticity:** Standard regression assumes the variance of your errors is constant. GLMs explicitly model situations where the variance changes with the mean (e.g., in count data, larger counts naturally have higher variance).

## Multiple Linear Regression

**Multiple Linear Regression (MLR)** extends simple linear regression by using **two or more independent variables** ($x_1, x_2, \dots, x_p$) to predict the outcome of a single continuous dependent variable ($y$).

Instead of fitting a straight line through a two-dimensional scatter plot, MLR fits a **hyperplane** (a flat surface) through a multi-dimensional space of data points.

The mathematical model is expressed as:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p + \epsilon$$

---

## Deconstructing the Intercept ($\beta_0$)

The intercept ($\beta_0$) represents the **expected value of the dependent variable ($y$) when all independent variables ($x$) are equal to zero**.

* **Geometric Meaning:** It is the precise point where the regression hyperplane intersects the $y$-axis.
* **Practical Interpretation:** In many real-world applications, the intercept serves as a baseline, though it may not always have a logical physical interpretation if a value of zero is impossible for the predictors (e.g., modeling a person's weight based on height and age—height cannot be zero).

---

## Deconstructing the Coefficients ($\beta_1, \beta_2, \dots, \beta_p$)

The coefficients (or slope parameters) represent the **ceteris paribus** ("all else being equal") relationships in the model. Each coefficient isolates the unique contribution of its respective predictor.

### 1. The Core Interpretation

A specific coefficient ($\beta_i$) quantifies the expected change in the dependent variable ($y$) for every **one-unit increase** in that specific independent variable ($x_i$), **assuming all other variables in the model are held strictly constant**.

For example, if you model an employee's salary ($y$) based on Years of Experience ($x_1$) and Number of Certifications ($x_2$), a $\beta_1$ value of $3,500$ means: *For two employees with the exact same number of certifications, each additional year of experience is associated with an average salary increase of $3,500.*

### 2. Controlling for Confounding

This "holding other variables constant" aspect is what makes MLR so powerful. It mathematically filters out the overlapping variance between correlated predictors, allowing you to estimate the **partial effect** of each variable.

### 3. Sign and Magnitude

* **Positive Coefficient ($+$):** A direct relationship. As $x_i$ increases, $y$ increases.
* **Negative Coefficient ($-$):** An inverse relationship. As $x_i$ increases, $y$ decreases.
* **Magnitude:** The raw scale of the coefficient depends entirely on the units of measurement for that specific $x_i$. Because of this, you cannot directly compare the raw magnitudes of two coefficients to determine which variable is "more important" unless the data has been standardized (z-scored) beforehand.

Would you like to explore how we evaluate the statistical significance of these individual coefficients using $t$-tests, or perhaps look at how multicollinearity can distort their interpretation?

## The Cost of Following Your Team: A Linear Regression Data Story

Here is a walk through of a **full data science process**, start to finish, using one running example: predicting how much it costs a fan to follow their national team through the FIFA World Cup 2026.

We touch every stage we've covered this semester:

* **Part 1 - The Data:** generate a realistic, deliberately messy dataset and save it to CSV
* **Part 2 - Exploratory Data Analysis (EDA):** understand the data before touching it
* **Part 3 - Data Cleaning:** constants, quasi-constants, duplicates, missing values, scaling, outliers
* **Part 4 - Feature Engineering:** derived features, categorical encoding (binary, one-hot, frequency)
* **Part 5 - Feature Selection:** correlation, VIF, SelectKBest, SelectFromModel, RFE, Random Forest importance
* **Part 6 - Modeling & Evaluation:** Linear Regression, coefficients, MSE, RMSE, MAE, R-squared, and a before/after comparison that tells the data story

Every code cell is followed by a short explanation of what just happened and why it matters. Use this notebook as your model for how to narrate your own final presentation - the goal isn't just correct code, it's a **clear story**: messy data in, disciplined process, clean signal out.

## The Scenario

It's summer 2026. The FIFA World Cup is being played across 16 host cities in the United States, Canada, and Mexico. Real reporting this year has put ticket face values anywhere from about \$60 to nearly \$11,000 depending on the match and seat category, and following one team all the way to the final has been estimated to cost $30,000 or more once flights, hotels, and daily spending are included.

Our dataset simulates 1,000+ fans, each of whom followed one national team through the tournament. Some fans' teams were eliminated in the group stage; others made a deep run. Importantly, **many fans stay in the host countries for the entire tournament window regardless of how their own team performs**, just to soak in the event - so total trip cost is driven by more than just how far your team advanced.

Our job: build a model that predicts **total trip cost in US dollars**, and tell a clean data story about which features actually drive that cost.

## Seed the Project

Keep track of your seed - if you ever need to reproduce your exact dataset, you'll need it.

In [ ]:
import time
import random
import numpy as np

def generate_user_seed():
    '''Builds a seed from the current time so each run gets unique data.'''
    nanoseconds = time.time_ns()
    random_component = random.randint(0, 1000)
    seed = (nanoseconds ^ random_component) % (2**32)
    return seed

random_state = generate_user_seed()
np.random.seed(random_state)
random.seed(random_state)
print('Your random seed for this run:', random_state)

**Explanation:** `generate_user_seed()` mixes the current nanosecond-resolution clock with a small random offset, then folds it into the 32-bit range NumPy's seed expects. Because it depends on *when* you run the cell, your dataset - and every downstream result: coefficients, MSE, everything - will be unique to you, even though your neighbor is running identical code.

# Part 1 - The Data

We're going to build this dataset in layers, the same way real messy data usually comes together: a core signal, buried under noise, redundancy, and imperfect record-keeping.

**The true (hidden) story we're building in:** total trip cost is driven by ticket spend, flight cost, hotel cost x nights stayed, and daily spending x nights stayed, plus some randomness. Everything else we add is either a *legitimate but non-causal* feature (categorical context), a *derived-later* feature (things that only become useful once we engineer them), or *pure noise and data-quality problems* that a real analyst would have to detect and handle.

## Install and Import

In [ ]:
!pip install Faker -q

**Explanation:** `Faker` generates realistic-looking fake personal data (names, countries, etc.). We'll use it only for flavor columns that make the dataset feel like real fan records - they won't be predictive of cost, and we'll drop them during Part 4.

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker

fake = Faker()
Faker.seed(random_state)

n = 1000
print(f'Generating data for {n} fans following a team through the FIFA World Cup 2026...')

**Explanation:** We seed `Faker` too, so the fake names are reproducible alongside everything else. `n = 1000` fans is our base sample size before we later append some duplicate rows.

## Helper Functions

These are the same kinds of utility functions we've used all semester to inject realistic data-quality problems: missing values, quasi-constant columns, and outliers. Read through them - you'll be asked to reason about what each one does when we get to Part 3.

In [ ]:
def gen_null(series, perc):
    '''Replaces `perc` percent of a Series' values with np.nan.'''
    var = series.copy()
    num_nulls = int(len(var) * (perc / 100))
    if num_nulls > 0:
        idx = np.random.choice(var.index, size=num_nulls, replace=False)
        var.loc[idx] = np.nan
    return var

def gen_quasi_constants(primary_label, secondary_label, size, variation_percentage=0.02):
    '''Returns an array that is almost entirely `primary_label`, with a small
    percentage of `secondary_label` mixed in - a column that looks constant
    at a glance but technically isn't.'''
    return np.random.choice(
        [primary_label, secondary_label],
        size=size,
        p=[1 - variation_percentage, variation_percentage]
    )

def gen_outliers(series, outlier_percentage=0.08, outlier_magnitude=3):
    '''Inflates a random subset of a numeric Series to simulate data-entry
    errors or surge-pricing spikes (e.g. a last-minute knockout-round booking).'''
    var = series.copy().astype(float)
    n_outliers = int(len(var) * outlier_percentage)
    if n_outliers > 0:
        idx = np.random.choice(var.index, size=n_outliers, replace=False)
        var.loc[idx] = var.loc[idx] * outlier_magnitude
    return var

print('Helper functions ready: gen_null, gen_quasi_constants, gen_outliers')

**Explanation:** `gen_null` swaps a percentage of a column's values for `NaN`. `gen_quasi_constants` builds a column that's overwhelmingly one value, with just enough of a second value to be *technically* variable - these sneak past a `nunique() == 1` check but still carry almost no information. `gen_outliers` multiplies a random slice of values, mimicking real surge-pricing or data-entry spikes.

## Fan Flavor Data (Faker)

In [ ]:
fan_first_name = [fake.first_name() for _ in range(n)]
fan_last_name = [fake.last_name() for _ in range(n)]
fan_home_country = [fake.country() for _ in range(n)]

df = pd.DataFrame({
    'fan_first_name': fan_first_name,
    'fan_last_name': fan_last_name,
    'fan_home_country': fan_home_country,
})
df.head()

**Explanation:** These three columns make each row feel like a real fan record. They will not be useful predictors of trip cost (a name tells you nothing about spending), so we're planting a realistic reason to practice *dropping identifier/flavor columns* in Part 4.

## Categorical Context Features

We generate the categorical columns first, because a few of our numeric costs (ticket spend, hotel cost) will depend on them - just like in real life, a knockout-round ticket costs more than a group-stage ticket, and a luxury hotel costs more per night than a budget one.

In [ ]:
host_countries = ['USA', 'Canada', 'Mexico']
df['host_country'] = np.random.choice(host_countries, size=n, p=[0.6, 0.2, 0.2])

stages = ['Group Stage', 'Round of 16', 'Quarterfinal', 'Semifinal/Final']
stage_p = [0.55, 0.25, 0.13, 0.07]  # most teams exit early; few make a deep run
df['team_stage_reached'] = np.random.choice(stages, size=n, p=stage_p)

df['hotel_tier'] = np.random.choice(['Budget', 'Luxury'], size=n, p=[0.7, 0.3])
df['purchase_channel'] = np.random.choice(['Official', 'Resale'], size=n, p=[0.65, 0.35])

df[['host_country', 'team_stage_reached', 'hotel_tier', 'purchase_channel']].head()

**Explanation:** Four categorical columns, each with a different number of labels on purpose:
- `host_country` - 3 labels (a good one-hot encoding candidate)
- `team_stage_reached` - 4 labels (also a one-hot candidate; this is how far the fan's *own team* advanced)
- `hotel_tier` - 2 labels (a binary map-to-0/1 candidate)
- `purchase_channel` - 2 labels (also binary: official face-value vs. resale marketplace)

In [ ]:
fifa_teams = [
    'Argentina', 'Brazil', 'France', 'England', 'Spain', 'Germany', 'Portugal',
    'Netherlands', 'Belgium', 'Italy', 'Croatia', 'Uruguay', 'Colombia', 'Mexico',
    'USA', 'Canada', 'Japan', 'South Korea', 'Morocco', 'Senegal', 'Nigeria',
    'Ghana', 'Egypt', 'Tunisia', 'Australia', 'Switzerland', 'Denmark', 'Poland',
    'Serbia', 'Austria', 'Ecuador', 'Chile', 'Peru', 'Costa Rica', 'Panama',
    'Jamaica', 'Saudi Arabia', 'Iran', 'Qatar', 'Ivory Coast', 'Cameroon',
    'Algeria', 'Norway', 'Sweden', 'Scotland', 'Wales', 'Ukraine', 'Turkey'
]
print(f'{len(fifa_teams)} teams available')
df['team_name'] = np.random.choice(fifa_teams, size=n)

host_cities = [
    'Atlanta', 'Boston', 'Dallas', 'Houston', 'Kansas City', 'Los Angeles',
    'Miami', 'Mexico City', 'Monterrey', 'Guadalajara', 'New York/New Jersey',
    'Philadelphia', 'San Francisco Bay Area', 'Seattle', 'Toronto', 'Vancouver'
]
print(f'{len(host_cities)} host cities available')
df['host_city'] = np.random.choice(host_cities, size=n)

**Explanation:** `team_name` (48 labels) and `host_city` (16 labels) are our **high-cardinality categorical features**. One-hot encoding either of these would blow up our feature count for very little benefit per label - this is exactly the situation where **frequency encoding** (replacing each label with how often it appears in the data) is the better call, which we'll do in Part 4.

## Core Numeric Features (the true cost drivers)

Now we generate the numeric features that will actually drive total trip cost, with some realistic structure built in: knockout-round tickets cost more, resale tickets cost more than official ones, and luxury hotels cost more per night than budget ones.

In [ ]:
stage_base_price = {
    'Group Stage': 850,
    'Round of 16': 1450,
    'Quarterfinal': 2300,
    'Semifinal/Final': 4200,
}
base_ticket = df['team_stage_reached'].map(stage_base_price)
ticket_noise = np.random.normal(0, 300, n)
resale_multiplier = np.where(df['purchase_channel'] == 'Resale', 1.35, 1.0)
ticket_spend_usd_true = np.clip((base_ticket + ticket_noise) * resale_multiplier, 60, None)

flight_cost_true = np.clip(np.random.normal(650, 220, n), 150, None)

hotel_base = np.where(df['hotel_tier'] == 'Luxury', 450, 140)
hotel_cost_true = np.clip(hotel_base + np.random.normal(0, 70, n), 60, None)

trip_duration_days = np.clip(np.random.normal(18, 9, n).round(), 3, 60)

daily_extra_spend_usd = np.clip(np.random.normal(110, 45, n), 20, None)

print('Core numeric features generated (true, pre-noise-injection values)')

**Explanation:** This is the causal backbone of our dataset:
- **`ticket_spend_usd`** depends on `team_stage_reached` (knockout rounds cost more, mirroring real 2026 pricing where finals ran into the thousands) and `purchase_channel` (resale costs ~35% more than official face value)
- **`flight_cost_usd`** and **`daily_extra_spend_usd`** are independent noisy costs
- **`hotel_cost_per_night`** depends on `hotel_tier`
- **`trip_duration_days`** is how many nights the fan stayed

Notice that hotel cost and daily spend are **per night** - the *total* lodging and daily-living cost depends on multiplying by `trip_duration_days`. We're not creating that multiplied feature yet on purpose - that's a derived feature we'll engineer in Part 4, and its absence is exactly why our "everything as-is" model in Part 6 will underperform.

## Multicollinear Features

Two more numeric features, each intentionally correlated with something we already generated - real multicollinearity, not just noise.

In [ ]:
num_matches_watched_total = np.clip(
    (trip_duration_days * 0.85 + np.random.normal(0, 2.5, n)).round(), 2, 30
)

distance_traveled_miles = np.clip(np.random.normal(2800, 1400, n), 100, None)

num_host_cities_visited = np.clip(
    (distance_traveled_miles / 380 + np.random.normal(0, 0.8, n)).round(), 1, 12
)

print('Correlation: trip_duration_days vs num_matches_watched_total =',
      round(np.corrcoef(trip_duration_days, num_matches_watched_total)[0, 1], 2))
print('Correlation: distance_traveled_miles vs num_host_cities_visited =',
      round(np.corrcoef(distance_traveled_miles, num_host_cities_visited)[0, 1], 2))

**Explanation:** `num_matches_watched_total` (total matches watched, including other teams' games) tracks `trip_duration_days` - the longer you stay, the more matches you catch. `num_host_cities_visited` tracks `distance_traveled_miles` - visiting more cities means covering more ground. These are believable real-world relationships, but statistically they create **multicollinearity**: two features carrying overlapping information, which we'll need to detect with VIF in Part 2/5.

## The Target: Total Trip Cost

In [ ]:
base_admin_fee = 500
noise = np.random.normal(0, 400, n)

total_trip_cost_usd = (
    base_admin_fee
    + ticket_spend_usd_true
    + flight_cost_true
    + hotel_cost_true * trip_duration_days
    + daily_extra_spend_usd * trip_duration_days
    + noise
)
total_trip_cost_usd = np.clip(total_trip_cost_usd, 800, None).round(2)

df['total_trip_cost_usd'] = total_trip_cost_usd
print(df['total_trip_cost_usd'].describe().round(2))

**Explanation:** This is our ground truth, built the way a real cost would actually accumulate: a flat admin fee, plus ticket spend, plus a flight, plus hotel cost **multiplied by** nights stayed, plus daily spend **multiplied by** nights stayed, plus random noise representing everything we can't observe (haggling, exchange rates, an unplanned side trip). In a real dataset you would never get to see this formula - we're building it here so that later, when we evaluate our model, we have a genuine yardstick for whether it recovered the real story.

## Question 1

We built the formula that generates total_trip_cost_usd ourselves, so we know the "true" answer. In a real project, you never get to see that formula — so what's the actual goal of modeling, if not to guess the exact formula back? Answer here (do not delete any text in this cell):

## Assign the Messy (Stored) Versions of Our Cost Columns

Now we store the columns a fan would actually report - and we deliberately corrupt two of them with outliers, simulating last-minute bookings and surge pricing.

In [ ]:
df['ticket_spend_usd'] = ticket_spend_usd_true
df['flight_cost_usd'] = gen_outliers(pd.Series(flight_cost_true), outlier_percentage=0.06, outlier_magnitude=3)
df['hotel_cost_per_night'] = gen_outliers(pd.Series(hotel_cost_true), outlier_percentage=0.08, outlier_magnitude=2.5)
df['trip_duration_days'] = trip_duration_days
df['daily_extra_spend_usd'] = daily_extra_spend_usd
df['num_matches_watched_total'] = num_matches_watched_total
df['distance_traveled_miles'] = distance_traveled_miles
df['num_host_cities_visited'] = num_host_cities_visited

df[['ticket_spend_usd', 'flight_cost_usd', 'hotel_cost_per_night']].describe().round(2)

**Explanation:** `flight_cost_usd` and `hotel_cost_per_night` are our two **features with injected outliers**. Because we generated the target from the *true* (pre-outlier) values, the outliers we just added are pure noise relative to the target - exactly like a real reporting error would be. A model trained on the raw, uncapped values will have those extreme points tugging on its coefficients; cleaning them in Part 3 should help.

## Noise-Only Features

A couple of columns with no real relationship to trip cost at all - included because real datasets always have some.

In [ ]:
df['favorite_pregame_meal'] = np.random.choice(
    ['Tacos', 'Burgers', 'Poutine', 'Pizza', 'BBQ'], size=n
)
df['num_people_in_travel_party'] = np.random.choice(
    [1, 2, 3, 4, 5, 6], size=n, p=[0.15, 0.35, 0.2, 0.15, 0.1, 0.05]
)
print('Noise-only features added')

**Explanation:** `favorite_pregame_meal` and `num_people_in_travel_party` are believable columns a survey might collect, but they carry essentially no signal about total cost. Part of feature selection is learning to recognize (statistically, not just by story) that a column doesn't help - even when it *sounds* plausible.

## Dedicated Scaling Demonstration Features

Two isolated columns built purely to practice scaling - one needs standardization, one needs min-max normalization.

In [ ]:
df['feature_needs_standard_scaling'] = np.random.normal(500, 150, n)
df['feature_needs_minmax_scaling'] = np.random.uniform(0, 10000, n)
df[['feature_needs_standard_scaling', 'feature_needs_minmax_scaling']].describe().round(2)

**Explanation:** `feature_needs_standard_scaling` is normally distributed on its own scale - a great candidate for `StandardScaler` (zero mean, unit variance). `feature_needs_minmax_scaling` is uniformly bounded between 0 and 10,000 - a great candidate for `MinMaxScaler` (rescale to [0, 1]). Neither is related to trip cost; they exist purely so we can practice the *mechanics* of scaling in isolation.

## Duplicate Columns

In [ ]:
df['ticket_spend_usd_copy'] = df['ticket_spend_usd']
df['host_country_copy'] = df['host_country']
print('Duplicate columns created: ticket_spend_usd_copy, host_country_copy')

**Explanation:** These two columns are **exact copies** of columns we already have. In a real pipeline, duplicate features can sneak in from merged data sources or repeated survey questions. They add zero new information but can quietly distort feature-importance rankings and coefficient interpretation if left in - we'll detect and drop them in Part 3.

## Missing Values

In [ ]:
exclude_from_nulls = [
    'ticket_spend_usd', 'host_country', 'ticket_spend_usd_copy',
    'host_country_copy', 'total_trip_cost_usd'
]
null_candidates = [c for c in df.columns if c not in exclude_from_nulls]

for col in null_candidates:
    perc = np.random.choice([0, 3, 5, 8, 12, 20])
    df[col] = gen_null(df[col], perc)

df.isnull().mean().mul(100).round(1).sort_values(ascending=False)

**Explanation:** We inject missing values into most columns at random rates. Notice we excluded `ticket_spend_usd` and `host_country` (and their duplicates) - if we nulled the originals *after* copying them, the "duplicate" columns would no longer be exact duplicates, and we want that duplicate-detection lesson to be clean in Part 3. In a real dataset you rarely get to choose this - here we're choosing it deliberately so the lesson stays sharp.

## Constant and Quasi-Constant Features

In [ ]:
df['tournament_edition'] = '2026 FIFA World Cup'
df['ticket_currency'] = 'USD'

df['used_official_fifa_app'] = gen_quasi_constants('Yes', 'No', n, variation_percentage=0.02)
df['travel_insurance_purchased'] = gen_quasi_constants('Yes', 'No', n, variation_percentage=0.03)

print(df[['tournament_edition', 'ticket_currency']].nunique())
print(df['used_official_fifa_app'].value_counts(normalize=True).round(3))
print(df['travel_insurance_purchased'].value_counts(normalize=True).round(3))

**Explanation:** `tournament_edition` and `ticket_currency` are **true constants** - every single row has the same value, so they carry zero information for modeling. `used_official_fifa_app` and `travel_insurance_purchased` are **quasi-constants** - technically they vary, but one label dominates so heavily (97-98%+) that they're nearly as useless as a true constant. Both problems look identical in a `.head()` call but require slightly different detection logic, which we'll practice in Part 3.

## Duplicate Rows

In [ ]:
dupes = df.iloc[0:12].copy()
df = pd.concat([df, dupes], axis=0).reset_index(drop=True)
print('New shape after adding duplicate rows:', df.shape)

**Explanation:** We copy the first 12 rows and append them to the bottom of the dataframe. Duplicate *rows* are a different problem from duplicate *columns* - they don't add new features, but they do let the same observation influence the model more than once, which can bias both statistics and model fit if left in.

## Shuffle Column Order

In [ ]:
cols = [c for c in df.columns if c != 'total_trip_cost_usd']
np.random.shuffle(cols)
df = df[cols + ['total_trip_cost_usd']]
print(f'Final shape: {df.shape}')
df.head()

**Explanation:** We shuffle every column except the target (kept last for readability) so students can't rely on column position to guess what's what - you have to actually look at each column's name, dtype, and values.

## Save to CSV

Check the folder icon on the left panel strip to see where your file is stored. Remember, the file is temporary.

In [ ]:
df.to_csv('fifa_2026_cost_of_following_a_team_pt1.csv', index=False)
print('Saved: fifa_2026_cost_of_following_a_team_pt1.csv')
print(df.shape)

**Explanation:** This is our Part 1 checkpoint. From here forward, every part reloads from the saved CSV rather than reusing the in-memory `df` - the same pattern you should use in your own projects, so each stage is reproducible and inspectable on its own.

# Part 2 - Exploratory Data Analysis (EDA)

Exploratory Data Analysis is how we get to know a dataset before we touch it. It's tempting to jump straight to modeling, but skipping this step means cleaning and feature engineering decisions get made blind. We reload from the CSV checkpoint to keep this part self-contained and reproducible.

## Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('fifa_2026_cost_of_following_a_team_pt1.csv')
print(df.shape)
print(df.info())
df.head()

**Explanation:** `df.info()` gives us row count, column count, dtypes, and non-null counts in one shot - it's the fastest way to spot which columns already look like they'll need attention (lots of missing values, object dtype where you expected numbers, etc.).

## Variable Types

Identifying variable types is the "measure twice, cut once" step of data science. You can't compute a meaningful mean on a hotel tier label, and you can't one-hot encode a dollar amount. Getting this right up front saves rework later.

In [ ]:
df_numerical = df.select_dtypes(include='number').columns
df_categorical = df.select_dtypes(include=['object']).columns
print('Numerical columns:', list(df_numerical))
print()
print('Categorical columns:', list(df_categorical))

**Explanation:** Everything numeric (dollar amounts, counts, our two isolated scaling-demo columns) falls into `df_numerical`. Everything text-labeled (host country, team name, hotel tier, our constants and quasi-constants) falls into `df_categorical`. Notice `tournament_edition` and `ticket_currency` show up here too, even though they don't vary - dtype alone won't tell us that they're constants.

## Target Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['total_trip_cost_usd'], kde=True, color='steelblue')
plt.title('Distribution of Total Trip Cost (USD)')
plt.xlabel('Total Trip Cost ($)')
plt.ylabel('Frequency')
plt.show()

**Explanation:** Trip cost is right-skewed - most fans cluster in a moderate cost range, with a long tail of expensive trips (deep tournament runs, resale tickets, luxury hotels). Right-skew in a regression target is worth noting: Linear Regression assumes roughly normal residuals, so a heavily skewed target can sometimes benefit from a log transform. We'll keep it in raw dollars here since that's the most interpretable unit for our story, but it's worth knowing this option exists.

## Question 2

The target is right-skewed. What decision might this affect later, and what would you look at to decide if a transformation is worth it? Answer here (do not delete any text in this cell):

## Categorical Feature Frequencies

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='team_stage_reached', data=df,
              order=['Group Stage', 'Round of 16', 'Quarterfinal', 'Semifinal/Final'])
plt.title('How Far Did Fans\' Teams Advance?')
plt.xticks(rotation=20)
plt.show()

**Explanation:** As designed, most fans' teams were eliminated in the group stage, with progressively fewer reaching each later round - mirroring how a single-elimination bracket actually thins out.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='team_stage_reached', y='total_trip_cost_usd', data=df,
            order=['Group Stage', 'Round of 16', 'Quarterfinal', 'Semifinal/Final'])
plt.title('Total Trip Cost by Team Stage Reached')
plt.xticks(rotation=20)
plt.show()

**Explanation:** Trip cost climbs with how far the team advanced - which makes sense, since knockout-round tickets cost more. But notice the boxes still overlap a lot: stage reached is *related* to cost, but far from the only driver, which is exactly the story we built in on purpose (remember, many fans stay the whole tournament regardless of their team's fate).

## Correlation Matrix

Correlation analysis tells us two things at once: which numeric features move together with our target (useful signal), and which numeric features move together with *each other* (multicollinearity - redundant signal that can destabilize coefficient estimates).

In [ ]:
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr().round(2)
corr['total_trip_cost_usd'].sort_values(ascending=False)

**Explanation:** This ranks every numeric column by how strongly it correlates with `total_trip_cost_usd`. `ticket_spend_usd` (and its exact duplicate) should sit near the top, since it feeds directly into the target formula. `feature_needs_standard_scaling`, `feature_needs_minmax_scaling`, and `num_people_in_travel_party` should sit near zero - they were built with no relationship to cost at all.

In [ ]:
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
# sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=False, square=True)
sns.heatmap(
    corr,
    mask=mask,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
)
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

**Explanation:** The heatmap gives us a visual shortcut to spotting multicollinearity: look for bright off-diagonal cells. You should see one between `trip_duration_days` and `num_matches_watched_total`, and another between `distance_traveled_miles` and `num_host_cities_visited` - the two multicollinear pairs we built in deliberately.

## Checking Multicollinearity Visually

In [ ]:
sns.jointplot(x='distance_traveled_miles', y='num_host_cities_visited', data=df, kind='reg', color='purple')
plt.suptitle('Distance Traveled vs. Number of Host Cities Visited', y=1.02)
plt.show()

**Explanation:** A tight, positive relationship confirms what the heatmap suggested - these two columns are largely telling us the same thing. Keeping both in a linear model can inflate the variance of their coefficient estimates, making the coefficients unstable and hard to trust (a small change in the data can flip their signs).

## Variance Inflation Factor (VIF)

VIF quantifies multicollinearity numerically: it measures how much a feature's variance is inflated because it's linearly predictable from the other features. A common rule of thumb is that VIF above 10 signals a problem worth addressing.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_check = numeric_df.drop(columns=['total_trip_cost_usd']).copy()
vif_check = vif_check.fillna(vif_check.mean())

vif_df = pd.DataFrame()
vif_df['feature'] = vif_check.columns
vif_df['VIF'] = [variance_inflation_factor(vif_check.values, i) for i in range(vif_check.shape[1])]
vif_df.sort_values('VIF', ascending=False)

**Explanation:** We temporarily mean-impute missing values just for this check, since VIF can't be computed with `NaN`s present - this is a scratch calculation, not our real cleaning step. Watch for `ticket_spend_usd` and `ticket_spend_usd_copy` (perfectly collinear - a duplicate column always produces enormous VIF), along with our two multicollinear pairs.

## Question 3

Two features being highly correlated doesn't necessarily hurt a model's predictions much — so why do we still care about multicollinearity? Answer here (do not delete any text in this cell):

## Outliers

Outliers are values that differ sharply from the rest of the distribution. Left unchecked, they can distort a linear model's coefficients (Linear Regression fits by minimizing squared error, so a few extreme points can dominate the loss function).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.boxplot(column='flight_cost_usd', ax=axes[0])
axes[0].set_title('Flight Cost ($)')
df.boxplot(column='hotel_cost_per_night', ax=axes[1])
axes[1].set_title('Hotel Cost per Night ($)')
plt.tight_layout()
plt.show()

**Explanation:** Both boxplots should show a cluster of points well above the upper whisker - the surge-pricing/data-entry outliers we injected in Part 1. These are our two designated **features with outliers**.

In [ ]:
def count_outliers_iqr(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < lower) | (s > upper)).sum()

for col in ['flight_cost_usd', 'hotel_cost_per_night', 'ticket_spend_usd', 'trip_duration_days']:
    print(f'{col}: {count_outliers_iqr(df[col])} outliers (IQR method)')

**Explanation:** The IQR method flags anything outside 1.5x the interquartile range beyond Q1/Q3. `flight_cost_usd` and `hotel_cost_per_night` should show noticeably more flagged outliers than the other two columns, confirming what the boxplots showed visually.

## Question 4

We chose to cap these outliers rather than delete the rows or leave them alone. What would you have lost or gained with each of those alternatives? Answer here (do not delete any text in this cell):

## Missing Data Summary

In [ ]:
missing_pct = df.isnull().mean().mul(100).round(1).sort_values(ascending=False)
missing_pct[missing_pct > 0]

**Explanation:** This lists every column with any missing data, worst first. Recall from Part 1 that `ticket_spend_usd`, `host_country`, and their duplicate columns were deliberately excluded from null injection - you should see them absent from this list (0% missing).

## Save EDA Checkpoint

Check the folder icon on the left panel strip to see where your file is stored. Remember, the file is temporary.

In [ ]:
df.to_csv('fifa_2026_cost_of_following_a_team_pt2.csv', index=False)
print('Saved: fifa_2026_cost_of_following_a_team_pt2.csv')

**Explanation:** Nothing changed about the data itself in Part 2 - we only looked at it - so this checkpoint is identical to Part 1's. Saving it anyway keeps our part-by-part pattern consistent and makes each notebook part independently runnable.

# Part 3 - Data Cleaning

Now that we understand the data, we clean it. We'll work through each problem type in the same order we introduced them in Part 1: constants, quasi-constants, duplicate rows, duplicate columns, missing values, scaling, and outliers.

## Load Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('fifa_2026_cost_of_following_a_team_pt2.csv')
print(df.shape)
print(df.info())
df.head()

## Drop Constant Features

A constant feature has exactly one unique value (ignoring missing data) - it can never help distinguish one row from another, so it contributes nothing to a model.

In [ ]:
constant_cols = [col for col in df.columns if df[col].nunique(dropna=True) == 1]
print('Constant columns found:', constant_cols)

df = df.drop(columns=constant_cols)
print('New shape:', df.shape)

**Explanation:** `nunique(dropna=True) == 1` catches any column where every non-missing value is identical - exactly `tournament_edition` and `ticket_currency`. Dropping them costs us nothing predictively and removes noise from anything downstream (correlation matrices, VIF, encoding).

## Drop Quasi-Constant Features

A quasi-constant feature *technically* has more than one value, but one value so overwhelmingly dominates that the column carries almost no usable variation. We flag these with a dominant-value-percentage threshold instead of `nunique()`.

In [ ]:
threshold = 0.95  # if one value makes up 95%+ of a column, treat it as quasi-constant

quasi_constant_cols = []
for col in df.select_dtypes(include=['object']).columns:
    top_freq = df[col].value_counts(normalize=True, dropna=True).iloc[0]
    if top_freq >= threshold:
        quasi_constant_cols.append(col)

print('Quasi-constant columns found:', quasi_constant_cols)
df = df.drop(columns=quasi_constant_cols)
print('New shape:', df.shape)

**Explanation:** This should catch `used_official_fifa_app` and `travel_insurance_purchased` - both built so one label dominates 95%+ of rows. The threshold (95% here) is a judgment call: too low and you'll drop features with real minority-class signal; too high and near-constant noise slips through. Always sanity-check the list before dropping.

## Drop Duplicate Rows

In [ ]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f'Dropped {before - after} duplicate rows. New shape: {df.shape}')

**Explanation:** `drop_duplicates()` removes rows that are identical across every remaining column. This should remove the 12 duplicate rows we appended in Part 1. Note: had we not dropped the constant/quasi-constant columns first, this would still work the same way, since those columns were identical across the duplicated rows anyway.

## Drop Duplicate Columns

A duplicate column is a different problem from a duplicate row: it's not about repeated observations, it's about the same information being stored twice under different names.

In [ ]:
duplicate_pairs = []
cols = list(df.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df[cols[i]].equals(df[cols[j]]):
            duplicate_pairs.append((cols[i], cols[j]))

# Column shuffling in Part 1 means either name in a pair could come first,
# so when dropping we keep whichever name looks like the "original" and
# drop the one that looks like the copy.
duplicate_cols = []
for col_1, col_2 in duplicate_pairs:
    duplicate_cols.append(col_2 if col_2.endswith('_copy') else col_1)

print('Duplicate column pairs found:', duplicate_pairs)
print('Dropping:', duplicate_cols)
df = df.drop(columns=duplicate_cols)
print('New shape:', df.shape)

**Explanation:** We compare every pair of columns for exact equality. This should catch `ticket_spend_usd_copy` and `host_country_copy` - exact copies of `ticket_spend_usd` and `host_country`. This pairwise comparison is O(n^2) in the number of columns, which is fine here, but for very wide datasets you'd want a faster approach (e.g. hashing each column).

## Handle Missing Data

We have a few standard options for missing data: drop rows, drop columns, or impute. With missingness scattered at moderate rates (mostly under 20%) across many columns, dropping rows would cost us a large fraction of the dataset, and dropping columns would throw away real signal. We'll impute instead: median for numeric columns (robust to the outliers we haven't handled yet), mode for categorical columns.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.drop('total_trip_cost_usd')
categorical_cols = df.select_dtypes(include='object').columns

for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print('Remaining missing values:', df.isnull().sum().sum())

**Explanation:** We use the **median** for numeric columns rather than the mean, specifically because we haven't dealt with outliers yet - the median is far less sensitive to the extreme surge-pricing values still sitting in `flight_cost_usd` and `hotel_cost_per_night`. Categorical columns get the **mode** (most frequent label), a standard simple imputation strategy. We deliberately excluded `total_trip_cost_usd` (our target) from imputation - you should never fabricate values for the thing you're trying to predict; if the target itself is missing, that row should typically be dropped instead.

## Question 5

We used median for numeric columns and mode for categorical ones. Under what circumstances would each of those choices be a bad idea? Answer here (do not delete any text in this cell):

## Handle Outliers

We cap (Winsorize) our two outlier-prone columns at the IQR fences rather than deleting the rows outright - this keeps the sample size intact while preventing a handful of extreme points from dominating the model fit.

In [ ]:
def cap_outliers_iqr(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    return series.clip(lower=lower, upper=upper)

before_max_flight = df['flight_cost_usd'].max()
before_max_hotel = df['hotel_cost_per_night'].max()

df['flight_cost_usd'] = cap_outliers_iqr(df['flight_cost_usd'])
df['hotel_cost_per_night'] = cap_outliers_iqr(df['hotel_cost_per_night'])

print(f"flight_cost_usd max: {before_max_flight:.2f} -> {df['flight_cost_usd'].max():.2f}")
print(f"hotel_cost_per_night max: {before_max_hotel:.2f} -> {df['hotel_cost_per_night'].max():.2f}")

**Explanation:** Capping (clipping every value to the IQR fence rather than removing the row) preserves every fan's other feature values while neutralizing the distortion an extreme dollar figure would cause in a squared-error loss function like Linear Regression's. This is one of several valid strategies - dropping outlier rows or log-transforming the column are others - but capping is a good default when you don't have strong evidence the outliers are simply invalid data.

## Scaling

`feature_needs_standard_scaling` and `feature_needs_minmax_scaling` are our two dedicated scaling-demo columns. We apply `StandardScaler` (zero mean, unit variance - the right choice when a feature is roughly normal and you care about relative distance from the mean) and `MinMaxScaler` (rescale to a fixed [0, 1] range - the right choice when a feature has natural, meaningful bounds) respectively.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

standard_scaler = StandardScaler()
df['feature_needs_standard_scaling'] = standard_scaler.fit_transform(df[['feature_needs_standard_scaling']])

minmax_scaler = MinMaxScaler()
df['feature_needs_minmax_scaling'] = minmax_scaler.fit_transform(df[['feature_needs_minmax_scaling']])

df[['feature_needs_standard_scaling', 'feature_needs_minmax_scaling']].describe().round(3)

**Explanation:** After `StandardScaler`, `feature_needs_standard_scaling` should have a mean near 0 and a standard deviation near 1. After `MinMaxScaler`, `feature_needs_minmax_scaling` should range from exactly 0 to exactly 1. In practice, you'd typically fit scalers on your training set only and apply the fitted transform to the test set (to avoid leaking test-set statistics) - we're scaling the full dataset here purely to demonstrate the mechanics; we'll respect train/test separation properly once we get to modeling in Part 5-6.

## Save Cleaning Checkpoint

In [ ]:
df.to_csv('fifa_2026_cost_of_following_a_team_pt3.csv', index=False)
print('Saved: fifa_2026_cost_of_following_a_team_pt3.csv')
print(df.shape)
df.info()

**Explanation:** Compare this shape to Part 1's 28 columns - we've already dropped 4 (2 constants, 2 quasi-constants) and 2 more (duplicate columns), plus removed the 12 duplicate rows. Missing values and outliers are handled, but the data still isn't ready for modeling: several columns are still text labels, and we haven't created any derived features yet. That's Part 4.

# Part 4 - Feature Engineering

With the data cleaned, we can build new features and get everything into a numeric form a model can use.

## Load Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('fifa_2026_cost_of_following_a_team_pt3.csv')
print(df.shape)
df.head()

## Derived Variables

Recall from Part 1: total trip cost depends on `hotel_cost_per_night` **multiplied by** `trip_duration_days`, and on `daily_extra_spend_usd` **multiplied by** `trip_duration_days`. A linear model can only combine features *additively* (a weighted sum) - it cannot discover a multiplicative interaction between two raw columns on its own. If we hand it `hotel_cost_per_night` and `trip_duration_days` separately, it has to approximate a curved relationship with a flat one, which it will do poorly. If we hand it the **product** directly, the relationship becomes linear again, and the model can fit it well. This is exactly why derived features matter.

In [ ]:
df['lodging_cost_total'] = df['hotel_cost_per_night'] * df['trip_duration_days']
df['daily_spend_total'] = df['daily_extra_spend_usd'] * df['trip_duration_days']

# A second style of derived feature: a rate, rather than a product -
# useful for context but not something the target formula depends on directly
df['distance_per_city'] = df['distance_traveled_miles'] / df['num_host_cities_visited'].replace(0, np.nan)

df[['lodging_cost_total', 'daily_spend_total', 'distance_per_city']].describe().round(2)

**Explanation:** `lodging_cost_total` and `daily_spend_total` are the two derived features we expect to matter most - they directly reconstruct the multiplicative terms baked into our target formula. `distance_per_city` is a different flavor of derived feature: a *rate* (miles per city visited) rather than a *total*. It's a reasonable engineering idea - it captures how spread out a fan's itinerary was - but since it wasn't part of the true cost formula, we shouldn't expect it to be nearly as predictive. Part of the exercise in Part 5 is letting the data show us that difference, rather than assuming every engineered feature is automatically useful.

## Question 6

Why can't a linear model learn the relationship hotel_cost_per_night × trip_duration_days on its own from the two raw columns? What does that tell you about the limits of the algorithm, not just the data? Answer here (do not delete any text in this cell):

## Categorical Encoding: Binary Features

`hotel_tier` and `purchase_channel` each have exactly two labels. We map these directly to 0/1 rather than one-hot encoding them - one-hot encoding a binary feature would just create a second, perfectly redundant column.

In [ ]:
df['hotel_tier'] = df['hotel_tier'].map({'Budget': 0, 'Luxury': 1})
df['purchase_channel'] = df['purchase_channel'].map({'Official': 0, 'Resale': 1})

df[['hotel_tier', 'purchase_channel']].head()

**Explanation:** We chose `Luxury -> 1` and `Resale -> 1` deliberately, so that a positive coefficient on either feature has an intuitive reading later ('being in the higher-cost category adds this many dollars'). The mapping direction is a modeling choice worth being deliberate about, since it directly affects how you'll narrate the coefficient's sign.

## Categorical Encoding: One-Hot Features

`host_country` (3 labels) and `team_stage_reached` (4 labels) have few enough categories that one-hot encoding is a good fit - each label becomes its own 0/1 column, with no artificial ordering implied between them.

In [ ]:
df = pd.get_dummies(df, columns=['host_country', 'team_stage_reached'], drop_first=True)

new_cols = [c for c in df.columns if c.startswith('host_country_') or c.startswith('team_stage_reached_')]
df[new_cols] = df[new_cols].astype(int)  # pandas returns bool dtype by default; cast to int for modeling
print('New one-hot columns:', new_cols)
df[new_cols].head()

**Explanation:** `drop_first=True` drops one label from each group as the reference category (e.g. 'USA' and 'Group Stage' become the implicit baseline). This avoids the **dummy variable trap** - if we kept every label as its own column, the group would sum to a constant 1 across every row, creating perfect multicollinearity with the intercept. Every remaining one-hot column's coefficient will later be interpreted *relative to that dropped baseline*.

## Categorical Encoding: Frequency Encoding

`team_name` (up to 48 labels) and `host_city` (16 labels) have too many categories for one-hot encoding to be practical - it would add dozens of mostly-empty columns. Instead, we replace each label with how often it appears in the dataset.

In [ ]:
team_freq = df['team_name'].value_counts(normalize=True)
df['team_name_freq'] = df['team_name'].map(team_freq)

city_freq = df['host_city'].value_counts(normalize=True)
df['host_city_freq'] = df['host_city'].map(city_freq)

df[['team_name', 'team_name_freq', 'host_city', 'host_city_freq']].head()

**Explanation:** Frequency encoding compresses a high-cardinality column into a single numeric column, which keeps our feature count manageable. The tradeoff: two *different* labels that happen to appear equally often get the exact same encoded value, so the model can no longer tell them apart by identity - only by how common they are. That's a reasonable tradeoff here, since neither `team_name` nor `host_city` was built to causally affect trip cost in our data story anyway.

## Drop Non-Predictive / Now-Redundant Columns

In [ ]:
drop_cols = ['fan_first_name', 'fan_last_name', 'fan_home_country', 'team_name', 'host_city']
df = df.drop(columns=drop_cols)
print('Dropped:', drop_cols)
print('New shape:', df.shape)

**Explanation:** `fan_first_name`, `fan_last_name`, and `fan_home_country` were flavor columns from Faker with no genuine relationship to cost - a name can't predict a dollar amount. `team_name` and `host_city` are dropped now that we've captured their information in frequency-encoded form; keeping the original text columns around would just leave non-numeric data a model can't consume.

## Verify Everything Is Numeric

In [ ]:
remaining_object_cols = df.select_dtypes(include='object').columns
print('Remaining non-numeric columns:', list(remaining_object_cols))
df.info()

**Explanation:** `favorite_pregame_meal` is likely the only text column left - we haven't encoded it yet on purpose. It's a good candidate for one-hot encoding (few labels, no natural order), so let's finish the job.

In [ ]:
df = pd.get_dummies(df, columns=['favorite_pregame_meal'], drop_first=True)
meal_cols = [c for c in df.columns if c.startswith('favorite_pregame_meal_')]
df[meal_cols] = df[meal_cols].astype(int)  # cast bool dummies to int for modeling
print('Remaining non-numeric columns:', list(df.select_dtypes(include='object').columns))
print('Final shape:', df.shape)

**Explanation:** Every column is numeric now. This is the point where a dataset is technically ready to feed into a scikit-learn model - though as we'll see in Part 5 and 6, 'technically ready' and 'ready to produce a good model' are not the same thing.

## Save Feature Engineering Checkpoint

In [ ]:
df.to_csv('fifa_2026_cost_of_following_a_team_pt4.csv', index=False)
print('Saved: fifa_2026_cost_of_following_a_team_pt4.csv')
print(df.shape)

**Explanation:** This checkpoint is our full, cleaned, fully-encoded feature set - everything a model could possibly use. In Part 5 we'll narrow this down using several feature selection techniques, and in Part 6 we'll compare a model trained on *this* full set against a model trained on the *selected* subset.

# Part 5 - Feature Selection

We now have a fully numeric, cleaned feature set - but not every feature in it is actually useful. This section walks through several different feature selection techniques side by side, so you can compare what each one flags.

## Load Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('fifa_2026_cost_of_following_a_team_pt4.csv')
print(df.shape)
df.head()

## Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['total_trip_cost_usd'])
y = df['total_trip_cost_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)
print(X_train.shape, X_test.shape)

**Explanation:** We split before doing any feature selection, and we'll only ever compute selection statistics and fit models on `X_train`. If we let information from `X_test` influence which features we keep, our later evaluation would be optimistic and misleading - a subtle form of data leakage that's easy to introduce by accident.

## Correlation with Target

In [ ]:
train_with_target = X_train.copy()
train_with_target['total_trip_cost_usd'] = y_train
corr_with_target = train_with_target.corr()['total_trip_cost_usd'].drop('total_trip_cost_usd')
corr_with_target.sort_values(ascending=False)

**Explanation:** The simplest possible selection signal: how strongly does each feature move with the target, on its own? `ticket_spend_usd`, `lodging_cost_total`, `daily_spend_total`, and `flight_cost_usd` should sit near the top - our real cost drivers, including the two derived features we engineered specifically to capture the multiplicative relationships. `feature_needs_standard_scaling`, `feature_needs_minmax_scaling`, `distance_per_city`, and `num_people_in_travel_party` should sit near zero, exactly as designed.

## Variance Inflation Factor (Revisited)

We saw VIF in Part 2 as a diagnostic. Now we use it as a selection tool: iteratively drop the single highest-VIF feature and recompute, until every remaining feature is below a threshold.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = X_train.copy()
removed_for_vif = []
threshold = 10

while True:
    vif_values = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = max(vif_values)
    if max_vif < threshold:
        break
    max_index = vif_values.index(max_vif)
    removed_for_vif.append(X_vif.columns[max_index])
    X_vif = X_vif.drop(columns=[X_vif.columns[max_index]])

print('Removed for high VIF:', removed_for_vif)
print('Remaining features:', X_vif.shape[1])

**Explanation:** This should remove one column from each multicollinear pair we built - `num_matches_watched_total` or `trip_duration_days`, and `num_host_cities_visited` or `distance_traveled_miles` - whichever comes out on top of the loop first. It may also flag `lodging_cost_total`, since it was built by multiplying two other features we kept (`hotel_cost_per_night` and `trip_duration_days`), which can itself create multicollinearity. If that happens, it's worth pausing on: a derived feature can be *more* predictive than its parents individually while also being *collinear* with them - a real tension between predictive power and coefficient stability that doesn't have a single right answer.

## SelectKBest

`SelectKBest` scores every feature independently against the target using a statistical test (here, `f_regression`, an F-test appropriate for a continuous target) and keeps the top `k`.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

k = 8
selector = SelectKBest(score_func=f_regression, k=k)
selector.fit(X_train, y_train)

kbest_features = X_train.columns[selector.get_support()]
print(f'Top {k} features by SelectKBest:')
print(list(kbest_features))

**Explanation:** Like the raw correlation ranking, `SelectKBest` evaluates each feature in isolation - it doesn't account for redundancy between features, so it may still keep both halves of a multicollinear pair if they're each individually strong. It's a fast first pass, not a final answer.

## SelectFromModel (Lasso)

Lasso (L1-regularized linear regression) has a useful property: it can shrink unhelpful coefficients all the way to exactly zero, effectively performing feature selection as a side effect of fitting the model.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

lasso_selector = SelectFromModel(estimator=Lasso(alpha=50, random_state=random_state))
lasso_selector.fit(X_train_scaled, y_train)

lasso_features = X_train.columns[lasso_selector.get_support()]
print('Features kept by Lasso:')
print(list(lasso_features))

**Explanation:** We scale first because Lasso's penalty is applied to the coefficients directly, and coefficients on differently-scaled features aren't comparable - an unscaled dollar-valued feature and an unscaled 0/1 feature would be penalized very unevenly. The `alpha` parameter controls how aggressively coefficients get pushed toward zero; a larger `alpha` keeps fewer features.

## Recursive Feature Elimination (RFE)

RFE fits a model, ranks features by importance, removes the weakest one, and repeats - unlike SelectKBest, this accounts for how features perform *together*, not just individually.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe_estimator = LinearRegression()
rfe_selector = RFE(rfe_estimator, n_features_to_select=8)
rfe_selector.fit(X_train_scaled, y_train)

rfe_features = X_train.columns[rfe_selector.get_support()]
print('Features kept by RFE:')
print(list(rfe_features))

**Explanation:** Because RFE re-fits the model at every elimination step, it can behave very differently from SelectKBest when features are correlated - it may keep only one member of a multicollinear pair, since once one is in the model, the other adds little marginal value.

## Random Forest Feature Importance

A completely different lens: rather than a linear model, we ask a tree-based ensemble which features it relies on most when splitting nodes to reduce prediction error.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

rf_selector = SelectFromModel(
    RandomForestRegressor(n_estimators=200, random_state=random_state),
    max_features=8
)
rf_selector.fit(X_train, y_train)

rf_features = X_train.columns[rf_selector.get_support()]
print('Features kept by Random Forest importance:')
print(list(rf_features))

**Explanation:** Random forests don't care about multicollinearity or feature scale the way linear models do, so this ranking can serve as a useful independent check against the linear-model-based methods above. If a feature shows up as important here *and* in the linear-based methods, that's a strong signal it genuinely matters.

## Bringing It Together

Each method above votes slightly differently. For our final model, we combine domain knowledge (we know the true cost formula) with what the data consistently pointed to: the direct cost components, plus a handful of context features, while dropping known noise, one member of each multicollinear pair, and duplicate-adjacent features.

In [ ]:
features_to_model = [
    'ticket_spend_usd',
    'flight_cost_usd',
    'lodging_cost_total',
    'daily_spend_total',
    'trip_duration_days',
    'hotel_tier',
    'purchase_channel',
    'team_name_freq',
]
features_to_model = [f for f in features_to_model if f in X_train.columns]
print('Final selected features:')
print(features_to_model)

X_train_selected = X_train[features_to_model]
X_test_selected = X_test[features_to_model]

**Explanation:** Notice what's missing on purpose: `num_matches_watched_total`/`num_host_cities_visited`/`distance_traveled_miles` (multicollinear with things we kept, or with each other), `distance_per_city` (a derived feature that never carried much signal), `feature_needs_standard_scaling`/`feature_needs_minmax_scaling` (built as pure noise), `num_people_in_travel_party` (no relationship to cost), and the one-hot host-country/team-stage columns (weaker signal than the direct cost components once ticket spend and lodging cost are already in the model). This is exactly the kind of judgment call you'll need to narrate in your own presentation: not just *which* features you kept, but *why*, backed by more than one selection method.

## Question 7

Every selection method gave a slightly different answer. If you were presenting to a non-technical stakeholder, how would you justify your final feature list in one sentence? Answer here (do not delete any text in this cell):

# Part 6 - Modeling, Coefficients, and the Data Story

This is where everything comes together. We'll fit two Linear Regression models on the exact same train/test split:

* **Model A - Everything:** every cleaned, encoded feature from Part 4, with no selection applied
* **Model B - Selected:** just the features we settled on in Part 5

Then we'll compare them on four metrics - MSE, RMSE, MAE, and R-squared - and look closely at what their coefficients tell us (and don't tell us) about what actually drives the cost of following a team.

## Load Data and Recreate the Split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

df = pd.read_csv('fifa_2026_cost_of_following_a_team_pt4.csv')

X = df.drop(columns=['total_trip_cost_usd'])
y = df['total_trip_cost_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

# Model A: every raw/cleaned/encoded feature EXCEPT the derived interaction
# features we engineered in Part 4 - the "kitchen sink, no engineering" baseline
derived_features = ['lodging_cost_total', 'daily_spend_total', 'distance_per_city']
X_all = X.drop(columns=derived_features)
X_train_all = X_train.drop(columns=derived_features)
X_test_all = X_test.drop(columns=derived_features)

# Model B: the selected + engineered feature set from Part 5
features_to_model = [
    'ticket_spend_usd', 'flight_cost_usd', 'lodging_cost_total', 'daily_spend_total',
    'trip_duration_days', 'hotel_tier', 'purchase_channel', 'team_name_freq',
]
features_to_model = [f for f in features_to_model if f in X_train.columns]

X_selected = X[features_to_model]
X_train_selected = X_train[features_to_model]
X_test_selected = X_test[features_to_model]

print('Model A (all raw features, no engineering):', X_train_all.shape[1], 'features')
print('Model B (selected + engineered features):', X_train_selected.shape[1], 'features')

**Explanation:** Using the same `random_state` as Part 5 means this split is identical to what we selected features against - no leakage. Model A gets *every* cleaned and encoded feature except the two derived interaction terms; Model B gets the small, engineered set we settled on in Part 5. That one difference - derived features or not - is the crux of the comparison we're about to run.

## Why We Evaluate with Cross-Validation, Not Just One Split

A single train/test split gives you exactly one estimate of generalization error - and that estimate depends partly on which rows happened to land in the test set. With a dataset this size, a single split can occasionally make either model look better or worse than it really is, purely by chance.

**K-fold cross-validation** fixes this by splitting the data into K roughly equal folds, training on K-1 of them and testing on the remaining one, then rotating through all K folds so every row gets used for testing exactly once. Averaging the metric across all K folds gives a far more reliable estimate of how a model will generalize than any single split can - which matters a great deal when the whole point of this section is to compare two models fairly.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

cv_mse_a = -cross_val_score(LinearRegression(), X_all, y, cv=kf, scoring='neg_mean_squared_error')
cv_mse_b = -cross_val_score(LinearRegression(), X_selected, y, cv=kf, scoring='neg_mean_squared_error')

cv_mae_a = -cross_val_score(LinearRegression(), X_all, y, cv=kf, scoring='neg_mean_absolute_error')
cv_mae_b = -cross_val_score(LinearRegression(), X_selected, y, cv=kf, scoring='neg_mean_absolute_error')

cv_r2_a = cross_val_score(LinearRegression(), X_all, y, cv=kf, scoring='r2')
cv_r2_b = cross_val_score(LinearRegression(), X_selected, y, cv=kf, scoring='r2')

print('Model A - 5-fold CV MSE per fold: ', cv_mse_a.round(0))
print('Model B - 5-fold CV MSE per fold: ', cv_mse_b.round(0))

**Explanation:** Each array shows the MSE from one of the five folds. Notice the fold-to-fold variation within a single model - that's exactly the noise a single train/test split is exposed to. Averaging across folds smooths that noise out, which is why we'll use these averages as our headline comparison.

## Model A: Linear Regression on All Raw Features (No Engineering)

In [ ]:
model_a = LinearRegression()
model_a.fit(X_train_all, y_train)
preds_a = model_a.predict(X_test_all)

mse_a = cv_mse_a.mean()
rmse_a = np.sqrt(mse_a)
mae_a = cv_mae_a.mean()
r2_a = cv_r2_a.mean()

print(f'Model A - Mean CV MSE:  {mse_a:,.2f}')
print(f'Model A - Mean CV RMSE: {rmse_a:,.2f}')
print(f'Model A - Mean CV MAE:  {mae_a:,.2f}')
print(f'Model A - Mean CV R2:   {r2_a:.4f}')

**Explanation:** Model A throws nearly every feature we have at the problem - all the noise columns, both multicollinear pairs, every categorical encoding - but *without* the engineered interaction terms. It has `hotel_cost_per_night`, `trip_duration_days`, and `daily_extra_spend_usd` available individually, but a linear model can only combine them additively, so it can't reconstruct the multiplicative cost relationship they actually have. We also fit it on our single train/test split (`model_a`) - we'll come back to that fit later, purely to inspect its coefficients.

## Model B: Linear Regression on Selected Features

In [ ]:
model_b = LinearRegression()
model_b.fit(X_train_selected, y_train)
preds_b = model_b.predict(X_test_selected)

mse_b = cv_mse_b.mean()
rmse_b = np.sqrt(mse_b)
mae_b = cv_mae_b.mean()
r2_b = cv_r2_b.mean()

print(f'Model B - Mean CV MSE:  {mse_b:,.2f}')
print(f'Model B - Mean CV RMSE: {rmse_b:,.2f}')
print(f'Model B - Mean CV MAE:  {mae_b:,.2f}')
print(f'Model B - Mean CV R2:   {r2_b:.4f}')

**Explanation:** Model B uses only the 8 features we settled on after combining correlation, VIF, SelectKBest, Lasso, RFE, and Random Forest importance - most of them the real cost-driving components from our data story, including the two derived features that capture the multiplicative hotel/daily-spend relationships. As with Model A, we also keep the single-split fit (`model_b`) around for the coefficient discussion below.

## Understanding the Metrics

Before comparing, here's what each metric actually tells us:

* **MSE (Mean Squared Error):** the average of the squared prediction errors. Because errors are squared, MSE is in *dollars-squared* - a large number that's hard to interpret directly, but it penalizes big misses much more heavily than small ones, which is useful when large errors are especially costly to get wrong.
* **RMSE (Root Mean Squared Error):** the square root of MSE, which brings the units back to plain dollars. RMSE is a good "typical error size" number, but because it's still derived from squared errors, a few very large misses can inflate it more than MAE would.
* **MAE (Mean Absolute Error):** the average of the *absolute* prediction errors, also in dollars. Unlike MSE/RMSE, MAE treats every dollar of error equally regardless of size, so it's more robust to a handful of extreme misses and often easier to explain to a non-technical audience ("on average, we're off by about $X").
* **R-squared (Coefficient of Determination):** the proportion of variance in the target that the model explains, on a scale that tops out at 1.0 (perfect prediction) and can go negative (worse than just guessing the mean every time). It's scale-free, which makes it useful for comparing models even when you're not thinking in raw dollars - but it doesn't tell you *how big* a typical error is in real-world terms, which is exactly what RMSE and MAE are for.

No single metric tells the whole story - that's why we report all four together, averaged across cross-validation folds rather than from one lucky (or unlucky) split.

## Side-by-Side Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model A (All Raw Features)': [mse_a, rmse_a, mae_a, r2_a],
    'Model B (Selected + Engineered)': [mse_b, rmse_b, mae_b, r2_b],
}, index=['MSE', 'RMSE', 'MAE', 'R2'])

comparison.round(4)

**Explanation:** Read down the RMSE and MAE rows first - they're in the same dollar units as trip cost, so they're the most intuitive gauge of "how far off is a typical prediction." A lower MSE/RMSE/MAE and a higher R-squared for Model B, averaged across all five folds, tells the story we set out to demonstrate: fewer, better-chosen, better-engineered features can outperform throwing every available column at the problem - and that conclusion holds up across folds, not just in one lucky split.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics_dollars = comparison.loc[['RMSE', 'MAE']]
metrics_dollars.plot(kind='bar', ax=axes[0])
axes[0].set_title('RMSE and MAE ($) — Lower is Better')
axes[0].set_ylabel('Dollars')
axes[0].tick_params(axis='x', rotation=0)

comparison.loc[['R2']].plot(kind='bar', ax=axes[1], color=['tab:blue', 'tab:orange'], legend=False)
axes[1].set_title('R-squared — Higher is Better')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Explanation:** Visualizing the comparison makes the story land for an audience faster than a table of numbers alone - this is the single chart worth building your final presentation slide around.

## Question 8

R² only dropped slightly between the two models, but RMSE and MAE improved by hundreds of dollars. If you only had time to report one metric to a client, which would you pick, and what would you be hiding by leaving the others out? Answer here (do not delete any text in this cell):

## Coefficients: What Model B Is Actually Telling Us

In [ ]:
coef_table_b = pd.DataFrame({
    'feature': features_to_model,
    'coefficient': model_b.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(f'Intercept: {model_b.intercept_:,.2f}')
coef_table_b

**Explanation:** Because none of these features were standardized, each raw coefficient has a direct dollar interpretation - *holding every other feature constant*:

* `ticket_spend_usd`, `flight_cost_usd`, `lodging_cost_total`, and `daily_spend_total` should each land close to a coefficient of about 1.0 - meaning one more dollar spent in that category adds roughly one more dollar to total trip cost. That's the model recovering our true cost formula almost exactly, which is exactly what we'd hope for given how we built the target.
* `hotel_tier` and `purchase_channel` are 0/1 features, so their coefficients represent a flat dollar shift for being in the "1" category (Luxury hotel, or Resale purchase) versus the baseline, holding everything else constant.
* Watch `team_name_freq` closely: because its raw values are tiny (roughly 0.01-0.03, since it's a proportion of 1,000 rows split across up to 48 teams), its coefficient will look enormous by comparison - even though we already know from Part 5 that this feature carries very little real signal. **This is the single most important lesson about coefficients in this notebook: raw coefficient size depends on the scale of the input feature, not just on how much the feature actually matters.** A coefficient of 50,000 on a feature that ranges from 0.01 to 0.03 can matter less in practice than a coefficient of 1.0 on a feature that ranges from \$60 to \$4,000.

## A Fairer Comparison: Standardized Coefficients

To compare features' *relative* importance fairly, we refit Model B on standardized versions of the same features (zero mean, unit variance). Now every coefficient answers the same question in the same units: "how many standard deviations does the prediction move for a one-standard-deviation change in this feature?"

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_selected_scaled = scaler.fit_transform(X_train_selected)

model_b_scaled = LinearRegression()
model_b_scaled.fit(X_train_selected_scaled, y_train)

standardized_coef_table = pd.DataFrame({
    'feature': features_to_model,
    'standardized_coefficient': model_b_scaled.coef_
}).sort_values('standardized_coefficient', key=abs, ascending=False)

standardized_coef_table

**Explanation:** Once every feature is on the same scale, `team_name_freq`'s influence shrinks down to roughly where it belongs, and `ticket_spend_usd`, `lodging_cost_total`, and `daily_spend_total` should rise to the top as the features that move the prediction most per standard deviation of change. This is the version of the coefficient table you'd actually want to show an audience if the question is "which factors matter most" rather than "how many dollars does each factor add."

## Why Did Model B Improve?

Three things separate Model A from Model B, and it's worth pulling them apart:

1. **Feature engineering, the biggest lever:** `lodging_cost_total` and `daily_spend_total` gave Model B direct access to the multiplicative relationships baked into the true cost formula - relationships that were sitting right there in Model A's raw ingredients (`hotel_cost_per_night`, `daily_extra_spend_usd`, `trip_duration_days`) but not in a form a *linear* model could combine correctly on its own. A linear model can only add weighted features together; it cannot discover on its own that cost = rate x nights. This is why Model B's MSE improvement is large and shows up consistently no matter whose randomly generated dataset you're running this notebook on - it isn't a coincidence of one particular train/test split.
2. **Multicollinearity removal:** Model A still carries both `trip_duration_days` and `num_matches_watched_total`, and both `distance_traveled_miles` and `num_host_cities_visited` - each pair carrying overlapping information. This doesn't necessarily hurt Model A's raw prediction accuracy much, but it makes its coefficients unstable and hard to trust, which matters once you try to *interpret* the model rather than just use it as a black box.
3. **Noise removal:** `feature_needs_standard_scaling`, `feature_needs_minmax_scaling`, `num_people_in_travel_party`, and the weaker categorical encodings add estimation variance to Model A without adding real signal. With a sample size in the thousands this effect is small on its own, but it compounds with the other two.

The takeaway: a clean, well-selected feature set mostly buys you *stability and interpretability*. A well-engineered feature set can buy you *accuracy a model literally could not reach otherwise* - no matter how much data you throw at it or how carefully you clean it. That's why feature engineering, not just feature selection, deserves a central place in your own project's data story.

## Question 9

Of the three reasons listed (engineering, multicollinearity, noise), which do you think would matter most on a dataset with only 50 rows instead of 1,000? Would your answer change? Answer here (do not delete any text in this cell):

# Closing the Data Story

We started with a plausible but messy dataset: constants, quasi-constants, duplicate rows and columns, missing values, unscaled and outlier-riddled numeric columns, and categorical features ranging from binary to nearly-unbounded in cardinality. None of that mess was random noise for its own sake - it mirrored exactly the kinds of problems real survey and transactional data carries.

Through a disciplined process - EDA, cleaning, feature engineering, and feature selection backed by multiple independent methods - we arrived at a small, interpretable set of features whose coefficients tell a coherent story: **the cost of following your team through the FIFA World Cup 2026 is driven overwhelmingly by ticket spend, flights, and the product of your hotel rate and daily spend with how long you stay** - not by which team you support, how many matches you happen to watch, or metadata that only sounds like it should matter.

That's the real deliverable of a data science project: not just a lower error number, but a model whose story you can explain, defend, and act on.

## Question 10

If a classmate's dataset told a slightly different story than yours (different top feature, different R²), does that mean one of you did something wrong? Answer here (do not delete any text in this cell):